## Data Collection

Historical 1-minute BTC/USDT perpetual futures data were obtained from **Binance** using the **Python Binance API**.  

The code connects to Binance’s futures API and downloads 1-minute candlestick data (`futures_historical_klines`) for the specified date range. Each candlestick includes:

- Open, High, Low, Close prices  
- Trading volume and quote volume  
- Number of trades  
- Taker buy volumes  

For this project, the data was configured to download:

- **Trading type:** USDT-M futures  
- **Symbol:** `BTCUSDT`  
- **Interval:** 1-minute (`1m`)  
- **Date range:** *TBD*  

The raw data were stored in a Pandas DataFrame, preserving all fields exactly as returned by Binance.  


In [6]:
from binance.client import Client
import pandas as pd
import datetime

# Binance API credentials
api_key = 'PASTE YOURS :)'
api_secret = 'PASTE YOURS :)'

client = Client(api_key, api_secret)

# Define the symbol for BTC/USDT pair
symbol = 'BTCUSDT'

# Define custom start and end time
start_time = datetime.datetime(2025, 10, 14, 0, 0, 0)
end_time = datetime.datetime(2025, 10, 21, 0, 0, 0)

klines = client.futures_historical_klines(symbol=symbol, interval=Client.KLINE_INTERVAL_1MINUTE, start_str=str(start_time), end_str=str(end_time))

# Convert the data into a pandas dataframe for easier manipulation
df_M = pd.DataFrame(klines, columns=['Open Time', 'Open', 'High', 'Low', 'Close', 'Volume', 'Close Time', 'Quote Asset Volume', 'Number of Trades', 'Taker Buy Base Asset Volume', 'Taker Buy Quote Asset Volume', 'Ignore'])


columns_to_convert = ['Open', 'High', 'Low', 'Close', 'Volume', 'Quote Asset Volume', 'Number of Trades', 'Taker Buy Base Asset Volume', 'Taker Buy Quote Asset Volume']

for col in columns_to_convert:
    df_M[col] = df_M[col].astype(float)

df_M

,Open Time,Open,High,Low,Close,Volume,Close Time,Quote Asset Volume,Number of Trades,Taker Buy Base Asset Volume,Taker Buy Quote Asset Volume,Ignore
0,1760400000000,115112.0,115127.5,115073.6,115073.6,74.748,1760400059999,8.603353e+06,1708.0,30.087,3.462883e+06,0
1,1760400060000,115073.6,115073.7,114992.6,115066.1,120.631,1760400119999,1.387456e+07,2826.0,70.107,8.063019e+06,0
2,1760400120000,115066.1,115115.5,115023.0,115035.5,112.965,1760400179999,1.300002e+07,2745.0,43.701,5.028939e+06,0
3,1760400180000,115035.5,115095.9,115000.0,115095.8,94.819,1760400239999,1.090675e+07,2189.0,44.400,5.107222e+06,0
4,1760400240000,115095.8,115107.9,115055.9,115056.0,92.417,1760400299999,1.063527e+07,1996.0,44.994,5.177681e+06,0
...,...,...,...,...,...,...,...,...,...,...,...,...
10076,1761004560000,110533.2,110540.0,110500.0,110500.1,26.840,1761004619999,2.966453e+06,987.0,2.081,2.300160e+05,0
10077,1761004620000,110500.0,110510.0,110490.4,110490.5,17.381,1761004679999,1.920570e+06,831.0,4.563,5.042075e+05,0
10078,1761004680000,110490.4,110495.3,110464.8,110469.9,50.592,1761004739999,5.589781e+06,1162.0,6.234,6.887041e+05,0
10079,1761004740000,110469.9,110475.0,110469.9,110469.9,10.668,1761004799999,1.178531e+06,451.0,5.234,5.782167e+05,0


## Data Preprocessing

After downloading the raw data, preprocessing was performed in Python (Pandas):

1. Converted numeric columns from strings to floats for analysis  
2. Converted timestamps to **EST/NYC** timezone  

*NOTE:*(All datasets were converted from UTC to EST timezone - Binance always provides historical data in UTC) 

3. Constructed three datasets for the analysis:
   - **Full dataset**  
   - **Weekday dataset** 
   - **New York session dataset** 
   

In [7]:
import pandas as pd
import datetime
import pytz

# Convert timestamps from UTC to New York timezone (EST) (handles DST automatically)
ny_tz = pytz.timezone('America/New_York')
df_M['Open Time'] = pd.to_datetime(df_M['Open Time'], unit='ms', utc=True).dt.tz_convert(ny_tz)
df_M['Close Time'] = pd.to_datetime(df_M['Close Time'], unit='ms', utc=True).dt.tz_convert(ny_tz)

# Construct datasets

# 1. Full dataset
df_full = df_M.copy()

# 2. Weekday dataset (exclude weekends only)
df_weekday = df_M[df_M['Open Time'].dt.weekday < 5]  # Monday=0, Sunday=6

# 3. New York session dataset (9:30–16:00 NY time)
session_start = datetime.time(9, 30)
session_end = datetime.time(16, 0)
df_ny_session = df_M[df_M['Open Time'].dt.time.between(session_start, session_end)]


# Check dataset sizes
print("Full dataset rows:", len(df_full))
print("Weekday dataset rows:", len(df_weekday))
print("NY session dataset rows:", len(df_ny_session))


Full dataset rows: 10081
Weekday dataset rows: 7201
NY session dataset rows: 2737


## Save Preprocessed Datasets

The three preprocessed datasets (full, weekday, and New York session) are saved as CSV files in the project folder.

In [8]:
import os

# Define the folder path
folder_path = "PASTE YOURS :)"
os.makedirs(folder_path, exist_ok=True)

# Dictionary of dataset names and corresponding DataFrames
datasets = {
    "BTCUSDT_full_dataset.csv": df_full,
    "BTCUSDT_weekday_dataset.csv": df_weekday,
    "BTCUSDT_NY_session_dataset.csv": df_ny_session
}

# Save all datasets using a loop
for filename, df in datasets.items():
    path = os.path.join(folder_path, filename)
    df.to_csv(path, index=False, sep=';')

print("All datasets saved successfully!")


All datasets saved successfully!
